# Sentence Reconstruction

The purpose of this project is to take in input a sequence of words corresponding to a random permutation of a given english sentence, and reconstruct the original sentence.

The otuput can be either produced in a single shot, or through an iterative (autoregressive) loop generating a single token at a time.


CONSTRAINTS:
* No pretrained model can be used.
* The neural network models should have less the 20M parameters.
* No postprocessing should be done (e.g. no beamsearch)
* You cannot use additional training data.


BONUS PARAMETERS:

A bonus of 0-2 points will be attributed to incentivate the adoption of models with a low number of parameters.

# Dataset

The dataset is composed by sentences taken from the generics_kb dataset of hugging face. We restricted the vocabolary to the 10K most frequent words, and only took sentences making use of this vocabulary.

In [1]:
!pip install datasets

Download the dataset

In [1]:
from datasets import load_dataset
from keras.layers import TextVectorization
import tensorflow as tf
import numpy as np

np.random.seed(42)
ds = load_dataset('generics_kb',trust_remote_code=True)['train']

2024-06-09 12:26:27.120425: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-09 12:26:27.120540: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-09 12:26:27.277355: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Generating train split:   0%|          | 0/1020868 [00:00<?, ? examples/s]

Filter row with length greater than 8.


In [8]:
ds = ds.filter(lambda row: len(row["generic_sentence"].split(" ")) > 8 )
corpus = [ '<start> ' + row['generic_sentence'].replace(","," <comma>") + ' <end>' for row in ds ]
corpus = np.array(corpus)

Filter:   0%|          | 0/1020868 [00:00<?, ? examples/s]

Create a tokenizer and Detokenizer

In [9]:
tokenizer=TextVectorization( max_tokens=10000, standardize="lower_and_strip_punctuation", encoding="utf-8",) #con il max prende le piu frequenti. ordina i token del vocab dal piu frequente al meno frequente
tokenizer.adapt(corpus)

class TextDetokenizer:
    def __init__(self, vectorize_layer):
        self.vectorize_layer = vectorize_layer
        vocab = self.vectorize_layer.get_vocabulary()
        self.index_to_word = {index: word for index, word in enumerate(vocab)}

    def __detokenize_tokens(self, tokens):
        def check_token(t):
          if t == 3:
            s="<start>"
          elif t == 2:
            s="<end>"
          elif t == 7:
            s="<comma>"
          else:
            s=self.index_to_word.get(t, '[UNK]')
          return s

        return ' '.join([ check_token(token) for token in tokens if token != 0])

    def __call__(self, batch_tokens):
       return [self.__detokenize_tokens(tokens) for tokens in batch_tokens]


detokenizer = TextDetokenizer( tokenizer )
sentences = tokenizer( corpus ).numpy()


Remove from corpus the sentences where any unknow word appears

In [10]:
mask = np.sum( (sentences==1) , axis=1) >= 1
original_data = np.delete( sentences, mask , axis=0)

In [ ]:
original_data.shape

(241236, 28)

Shuffle the sentences

In [ ]:
# shifted_data = np.roll(original_data, 1, axis=1)

In [105]:
from tensorflow.keras.utils import Sequence

class DataGenerator(Sequence):
    def __init__(self, data, training=True, batch_size=32, shuffle=True):

        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.training = training
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        data_batch = np.array([self.data[k] for k in indexes])
        # shifted_result = np.array([shifted_data[k] for k in indexes])

        # copy of ordered sequences
        result = np.copy(data_batch)
        shifted_result = np.roll(result, -1, axis=1)

        #shuffle only the relevant positions for each batch
        for i in range(data_batch.shape[0]):
          np.random.shuffle(data_batch[i,1:data_batch[i].argmin() - 1])

        if(self.training):
          return (data_batch, result), shifted_result
        else:
          return (data_batch, data_batch), result

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
# train_generator = DataGenerator(original_data[:220000])
# test_generator = DataGenerator(original_data[220000:])
# x, y = test_generator.__getitem__(1)
# x = detokenizer(x[0])
# y = detokenizer(y)

# for i in range(7):
#   print("original: ", y[i])
#   print("shuffled: ", x[i])
#   print("\n")

original:  wind speed is a reflection of the air pressure gradients <end> <start>

shuffled:  <start> the speed pressure reflection of a air wind gradients is <end>





original:  visible light makes up a fraction of all electromagnetic energy <end> <start>

shuffled:  <start> up visible light of makes fraction electromagnetic energy a all <end>





original:  most women experience a menstrual period four to six weeks after a miscarriage <end> <start>

shuffled:  <start> six to menstrual miscarriage women a most after weeks experience a four period <end>





original:  all weasels have scent glands <comma> but some are more powerful than others <end> <start>

shuffled:  <start> more powerful but some glands have weasels <comma> scent all are others than <end>





original:  some women can develop the technique of achieving ejaculation <end> <start>

shuffled:  <start> achieving can of the ejaculation women develop technique some <end>





original:  yoga improves fitness <comma> l

# Metrics

Let s be the source string and p your prediction. The quality of the results will be measured according to the following metric:

1.  look for the longest substring w between s and p
2.  compute |w|/max(|s|,|p|)

If the match is exact, the score is 1.

When computing the score, you should NOT consider the start and end tokens.



The longest common substring can be computed with the SequenceMatcher function of difflib, that allows a simple definition of our metric.

In [13]:
from difflib import SequenceMatcher

def score(s,p):
  match = SequenceMatcher(None, s, p).find_longest_match()
  return (match.size/max(len(p),len(s)))

Let's do an example.

In [14]:
original = "at first henry wanted to be friends with the king of france"
generated = "henry wanted to be friends with king of france at the first"
print("your score is ",score(original,generated))

your score is  0.5423728813559322


The score must be computed as an average of at least 3K random examples taken form the test set.

# What to deliver

You are supposed to deliver a single notebook, suitably commented.
The notebook should describe a single model, although you may briefly discuss additional attempts you did.

The notebook should contain a full trace of the training.
Weights should be made available on request.

You must also give a clear assesment of the performance of the model, computed with the metric that has been given to you.

# Good work!

In [95]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [15]:
from keras.layers import LayerNormalization, MultiHeadAttention, Add, Conv1D, Input, Dense, Embedding
from keras.layers import GlobalAveragePooling1D, GlobalMaxPooling1D, Concatenate, LayerNormalization, Dropout
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

In [ ]:
def clean_sentence(x):
    x = x.replace('<start>', '').replace('<end>', '').replace('<pad>', '').strip()
    return x

In [16]:
### Positional Encoding ###
def positional_encoding(max_sequence_len, model_dim):
    """
    :param max_sequence_len: the maximum length of the sequence
    :param model_dim: the embedding dimension
    """

    positions = np.arange(max_sequence_len)[:, np.newaxis]
    dimensions = np.arange(model_dim)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(model_dim))
    angle_rads = positions * angle_rates

    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]
    return pos_encoding

In [17]:
### Transformer Block ###
def transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate):
    """
    :param latent_dim: the embedding dimension
    :param num_heads: the number of heads in the multiheadattention models
    :param feed_forward_dim: the number of neurons in the feedforward network
    :param dropout_rate: the dropout rate
    """

    input_layer = Input(shape=(None, latent_dim))

    # Multi-head attention
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    attention_output = attention(input_layer, input_layer, input_layer)
    attention_output = Dropout(dropout_rate)(attention_output)
    attention_output = Add()([input_layer, attention_output])
    attention_output = LayerNormalization()(attention_output)

    # Feedforward
    outputs = Dense(feed_forward_dim, activation='relu')(attention_output)
    outputs = Dense(latent_dim)(outputs)
    outputs = Dropout(dropout_rate)(outputs)
    outputs = Add()([attention_output, outputs])
    outputs = LayerNormalization()(outputs)

    return Model(input_layer, outputs)

In [18]:
### Transformer Model ###
def transformer_model(latent_dim, num_heads, feed_forward_dim, dropout_rate, max_sequence_len, num_transformer_blocks, vocab_size):
    """
    :param latent_dim: the embedding dimension
    :param num_heads: the number of heads in the multiheadattention models
    :param feed_forward_dim: the number of neurons in the feedforward network
    :param dropout_rate: the dropout rate
    :param max_sequence_len: the maximum sequence length
    :param num_transformer_blocks: the number of transformer blocks
    :param vocab_size: the size of the vocabulary
    """

    # Encoder
    encoder_inputs = Input(shape=(max_sequence_len,))
    encoder_embedding = Embedding(vocab_size, latent_dim)(encoder_inputs)
    # pos_encoding = positional_encoding(max_sequence_len, latent_dim)
    # encoder_outputs = encoder_embedding + pos_encoding
    encoder_outputs = encoder_embedding

    for i in range(num_transformer_blocks):
        encoder_outputs = transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)(encoder_outputs)

    # Decoder
    decoder_inputs = Input(shape=(max_sequence_len,))
    decoder_embedding = Embedding(vocab_size, latent_dim)(decoder_inputs)
    pos_encoding = positional_encoding(max_sequence_len, latent_dim)
    decoder_outputs = decoder_embedding + pos_encoding

    for i in range(num_transformer_blocks):
        decoder_outputs = transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)(decoder_outputs)

    # Attention
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    context_vector = attention(decoder_outputs, encoder_outputs, encoder_outputs)
    decoder_combined_context = Concatenate()([context_vector, decoder_outputs])
    
    # Re-adapt dimension
    decoder_combined_context = Dense(latent_dim, activation='relu')(decoder_combined_context) 

    # Transformer block
    decoder_outputs = transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)(decoder_combined_context)

    # Output layer
    decoder_outputs = Dense(vocab_size, activation='softmax')(decoder_outputs)
    
    return Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [163]:
latent_dim = 256
num_heads = 32
feed_forward_dim = 256
dropout_rate = 0.5
max_sequence_len = 28
num_transformer_blocks = 8
vocab_size = len(tokenizer.get_vocabulary())

model = transformer_model(latent_dim, num_heads, feed_forward_dim, dropout_rate, max_sequence_len, num_transformer_blocks, vocab_size)
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy')
model.summary()

Model: "functional_71"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_28      │ (None, 28)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 28, 256)   │  2,560,000 │ input_layer_28[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_19      │ (None, 28)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_51 (Add)        │ (None, 28, 256)   │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 28, 256)   │  2,560,000 │ input_layer_19[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_53       │ (None, 28, 256)   │    395,776 │ add_51[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_37       │ (None, 28, 256)   │    395,776 │ embedding_2[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_55       │ (None, 28, 256)   │    395,776 │ functional_53[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_39       │ (None, 28, 256)   │    395,776 │ functional_37[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_57       │ (None, 28, 256)   │    395,776 │ functional_55[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_41       │ (None, 28, 256)   │    395,776 │ functional_39[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_59       │ (None, 28, 256)   │    395,776 │ functional_57[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_43       │ (None, 28, 256)   │    395,776 │ functional_41[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_61       │ (None, 28, 256)   │    395,776 │ functional_59[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_45       │ (None, 28, 256)   │    395,776 │ functional_43[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_63       │ (None, 28, 256)   │    395,776 │ functional_61[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_47       │ (None, 28, 256)   │    395,776 │ functional_45[0]… │
│ (Functional)        │                   │            │                 

 Total params: 14,812,688 (56.51 MB)

 Trainable params: 14,812,688 (56.51 MB)

 Non-trainable params: 0 (0.00 B)

In [164]:
train_generator = DataGenerator(original_data[:220000], training=True, batch_size=64)
test_generator = DataGenerator(original_data[220000:], training=False, batch_size=64)

In [ ]:
# FIRST TRAINING - 5 EPOCHS ####
# checkpoint_path = "/content/drive/MyDrive/Colab/PRJ/transformer_data_data.keras"
# checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

history1 = model.fit(train_generator, epochs=5)
model.save('v1_bigger.keras')

Epoch 1/5


/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


   1/3437 ━━━━━━━━━━━━━━━━━━━━ 148:56:55 156s/step - loss: 9.2250

W0000 00:00:1717948875.173382     120 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


3437/3437 ━━━━━━━━━━━━━━━━━━━━ 449s 85ms/step - loss: 4.0919
Epoch 2/5
3437/3437 ━━━━━━━━━━━━━━━━━━━━ 293s 85ms/step - loss: 3.9973
Epoch 3/5
1236/3437 ━━━━━━━━━━━━━━━━━━━━ 3:07 85ms/step - loss: 3.9946

In [98]:
from IPython.display import FileLink
FileLink('v1_ddr.keras')

/kaggle/working/v1.keras

In [106]:
### SECOND TRAINING - 5 EPOCHS ###
from tensorflow.keras.models import load_model

history2 = model.fit(train_generator, epochs=5)
# model.save('/content/drive/MyDrive/Colab/PRJ/transformer8_128_32_8_10epochs.h5')

Epoch 1/5


/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


   1/3437 ━━━━━━━━━━━━━━━━━━━━ 124:54:00 131s/step - loss: 17.7299

W0000 00:00:1717942584.818601     119 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


3437/3437 ━━━━━━━━━━━━━━━━━━━━ 356s 65ms/step - loss: 4.1195
Epoch 2/5
3437/3437 ━━━━━━━━━━━━━━━━━━━━ 224s 65ms/step - loss: 3.9926
Epoch 3/5
3437/3437 ━━━━━━━━━━━━━━━━━━━━ 223s 65ms/step - loss: 3.9891
Epoch 4/5
3437/3437 ━━━━━━━━━━━━━━━━━━━━ 224s 65ms/step - loss: 3.9897
Epoch 5/5
3437/3437 ━━━━━━━━━━━━━━━━━━━━ 223s 65ms/step - loss: 3.9888


In [36]:
model = load_model('v1.keras')

In [ ]:
### Predict One Token at a Time ###
def predict(encoder_input, initial_state, model, max_sequence_len=28):
    
    batch_size = encoder_input.shape[0]
    decoder_input = np.zeros((batch_size, max_sequence_len))
    bow = [[word for word in sentence if word not in [3, 2, 0]] for sentence in encoder_input]

    for i in range(batch_size):
        decoder_input[i,0] = 3 # <start> token
        # decoder_input[i,27] = 2 # <end> token
    
#     output = model.predict([encoder_input, initial_state])
#     token = np.argmax(output[:, 1, :], axis=-1)
#     for j in range(len(token) - 1):
#         decoder_input[j,1] = token[j]

    for i in range(1, max_sequence_len):
        output = model.predict([encoder_input, decoder_input])

        # i-th word predicted, for each sentence of the batch
        # token = np.argmax(output[:, i-1, :], axis=-1)
        token = output
        token = token[:, -1, :]
 
        # add new word to sentence
        for j in range(len(token)):
            
            if len(bow[j]) == 0:
                cand_token = 2
            else:
            # choose index with highest score
                s_pred = token[j, np.array(bow[j])]
                cand_index = np.argmax(s_pred)
                cand_token = bow[j][cand_index]
                del bow[j][cand_index]
            
            decoder_input[j,i] = cand_token
            
            # decoder_input[j,i] = token[j]
        # print(decoder_input[0])
    return decoder_input

In [ ]:
### Show Predicted Sentences ###
x, y_true = test_generator.__getitem__(1)
y_pred = predict(x[0], x[1], model)

for true, pred in zip(y_true, y_pred):
    s_pred = clean_sentence ( detokenizer([pred])[0] )
    s_true = clean_sentence ( detokenizer([true])[0] )
    
    print("Predicted: ", s_pred)
    print("True: ", s_true)
    print('\n')

In [65]:
import random
def evaluate_baseline(test_generator, score_func):
    scores = []

    for i in range(len(test_generator)):
        x, y_true = test_generator.__getitem__(i)
        shuffled = x[0].copy()
        random.shuffle(shuffled)

        # loop because each item is a batch
        for true, shuffle in zip(detokenizer(y_true), detokenizer(shuffled)):
            scores.append(score_func(true, shuffle))
    return np.mean(scores), np.std(scores)

baseline_mean, baseline_std = evaluate_baseline(test_generator, score)
print (f'Baseline score: {baseline_mean}')
print (f'Baseline Stdev: {baseline_std}')
print (f'Baseline 3*Stdev: {baseline_std * 3}')

Baseline score: 0.09294867972837931
Baseline Stdev: 0.028445817866771132
Baseline 3*Stdev: 0.08533745360031339


In [162]:
def evaluate_model(model, test_generator, score_func):
    scores = []

    for i in range(len(test_generator)):
        print("Test :", i)
        x, y_true = test_generator.__getitem__(i)
        y_pred = predict(x[0], x[1], model)
        
        for true, pred in zip(y_true, y_pred):
            s_pred = clean_sentence ( detokenizer([pred])[0] )
            s_true = clean_sentence ( detokenizer([true])[0] )
            
            scores.append(score_func(s_true, s_pred))
    return np.mean(scores), np.std(scores)

average_score, stdev_score = evaluate_model(model, test_generator, score)
print(f'Average score: {average_score}')
print(f'Stdev: {stdev_score}')
print(f'3*Stdev: {stdev_score * 3}')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━

In [40]:
# improvement over random guess
improvement = (average_score - baseline_mean) / baseline_mean
print(f'Improvement over random guess: {improvement}')

Improvement over random guess: 0.5402242730425154
